\# Multi-Agent Customer Support System using Google Antigravity SDK

This notebook demonstrates a **Multi-Agent Customer Support System** built using the official **Google Antigravity SDK** (`google-antigravity`) and the `gemini-2.5-flash` model.

### System Architecture
The system orchestrates two specialized agents in sequence:
1. **Support Agent**: Equipped with a custom tool to fetch clean text from a webpage. It retrieves information from the specified URL and drafts a customer support response to answer the query.
2. **Reviewer Agent (Quality Check)**: Evaluates the draft response written by the Support Agent. It checks for factual accuracy against the user's query, tone (politeness/professionalism), spelling, and formatting, outputting a polished, final approved response.

## Step 1: Install Dependencies
First, we install the Google Antigravity SDK along with `requests` and `beautifulsoup4` for webpage fetching.

In [ ]:
!pip install google-antigravity beautifulsoup4 --trusted-host pypi.org --trusted-host files.pythonhosted.org --trusted-host pypi.python.org
!pip install --upgrade protobuf --trusted-host pypi.org --trusted-host files.pythonhosted.org --trusted-host pypi.python.org

## Step 1.5: Setup Glibc Compatibility (For Google Colab / Ubuntu 22.04)
The `google-antigravity` package bundles a compiled Go binary (`localharness`) that requires glibc 2.36 or higher.
Since Google Colab runs Ubuntu 22.04 LTS (which ships with glibc 2.35), running the binary directly raises a compatibility error.
This cell automatically downloads the compatible glibc dynamic linker and library files from Ubuntu 24.04 (Noble) and configures a local wrapper.

In [ ]:
import os
import sys
import shutil
import subprocess

if sys.platform.startswith("linux"):
    print("Detected Linux environment. Setting up glibc compatibility workaround...")

    # Download compatible libc6 deb package (contains glibc 2.39 with GLIBC_ABI_DT_RELR support)
    deb_url = "http://archive.ubuntu.com/ubuntu/pool/main/g/glibc/libc6_2.39-0ubuntu8_amd64.deb"
    deb_path = "/tmp/libc6.deb"
    extracted_dir = "/tmp/libc6-extracted"
    compat_dir = os.path.expanduser("~/.local/lib/glibc-compat")

    try:
        # Download deb
        print("Downloading compatible libc6 package...")
        subprocess.run(["wget", "-q", deb_url, "-O", deb_path], check=True)
        # Extract deb
        os.makedirs(extracted_dir, exist_ok=True)
        print("Extracting package...")
        subprocess.run(["dpkg-deb", "-x", deb_path, extracted_dir], check=True)
        # Copy libc.so.6 and ld-linux-x86-64.so.2
        os.makedirs(compat_dir, exist_ok=True)

        extracted_lib_dir = os.path.join(extracted_dir, "lib", "x86_64-linux-gnu")
        if not os.path.exists(extracted_lib_dir):
            extracted_lib_dir = os.path.join(extracted_dir, "usr", "lib", "x86_64-linux-gnu")

        shutil.copy(os.path.join(extracted_lib_dir, "libc.so.6"), compat_dir)
        shutil.copy(os.path.join(extracted_lib_dir, "ld-linux-x86-64.so.2"), compat_dir)

        # Create wrapper script
        wrapper_path = os.path.join(compat_dir, "localharness-wrapper")
        wrapper_content = f"""#!/bin/bash
REAL_HARNESS=$(python -c "import google.antigravity, os; print(os.path.join(os.path.dirname(google.antigravity.__file__), 'bin', 'localharness'))")
exec {compat_dir}/ld-linux-x86-64.so.2 --library-path {compat_dir} "$REAL_HARNESS" "$@"
"""
        with open(wrapper_path, "w") as f:
            f.write(wrapper_content)
        os.chmod(wrapper_path, 0o755)

        # Set environment variable for active session
        os.environ["ANTIGRAVITY_HARNESS_PATH"] = wrapper_path
        print(f"Glibc compatibility workaround configured successfully! Harness path set to: {wrapper_path}")

        # Cleanup
        shutil.rmtree(extracted_dir, ignore_errors=True)
        if os.path.exists(deb_path):
            os.remove(deb_path)
    except Exception as e:
        print(f"Warning: Failed to set up glibc compatibility workaround: {e}")
        print("The agent might fail to run if the host glibc version is older than 2.36.")
else:
    print("Non-Linux platform detected; skipping glibc workaround.")

Detected Linux environment. Setting up glibc compatibility workaround...
Extracting package...
Glibc compatibility workaround configured successfully! Harness path set to: /root/.local/lib/glibc-compat/localharness-wrapper


## Step 2: Configure Gemini API Key
To use Gemini models, you must provide a Gemini API Key. This cell checks Google Colab Secrets (named `GEMINI_API_KEY`) and falls back to prompting you directly if it is not set.

In [ ]:
import os
import getpass

from google.colab import userdata

os.environ['GEMINI_API_KEY'] = userdata.get('GOOGLE_API_KEY')

In [ ]:
from google.colab import auth
auth.authenticate_user()

## Step 3: Define Webpage Fetching Tool
We implement a clean text scraper `fetch_webpage_content` that handles HTTP requests, decomposes non-content tags (like scripts/styles), and limits output text to keep context token counts low.

In [ ]:
import requests
from bs4 import BeautifulSoup

def fetch_webpage_content(url: str) -> str:
    """
    Fetches the cleaned text content of a webpage given its URL.

    Args:
        url: The absolute URL of the webpage to fetch.

    Returns:
        The cleaned text content of the webpage (up to 5000 characters), or an error message.
    """
    try:
        # User-Agent header to avoid simple scraping blocks
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # Remove navigation, scripting, and style elements to isolate main content
        for element in soup(["script", "style", "nav", "header", "footer", "form"]):
            element.decompose()

        text = soup.get_text(separator="\n")

        # Clean extra whitespace
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        cleaned_text = "\n".join(chunk for chunk in chunks if chunk)

        # Limit to 5000 characters to keep context window efficient
        return cleaned_text[:5000]
    except Exception as e:
        return f"Error fetching webpage: {str(e)}"

## Step 4: Configure the Agents
We configure both agents using the `LocalAgentConfig` settings:
* **Support Agent**: Registered with the `fetch_webpage_content` tool and `policy.allow_all()` to enable autonomous tool use.
* **Reviewer Agent**: Programmed to critique and refine the writer's draft to output the final approved answer.

In [ ]:
from google.antigravity import Agent, LocalAgentConfig
from google.antigravity.hooks import policy

# Configure the Support Agent with the fetch tool
writer_config = LocalAgentConfig(
    system_instructions=(
        "You are a helpful customer support Agent.\n"
        "Your task is to draft a customer support response to the user's query.\n"
        "You MUST use the `fetch_webpage_content` tool to read the webpage at the provided URL "
        "and gather the necessary facts to answer the user's query accurately.\n"
        "Draft a complete, polite, and helpful response. Answer only using facts from the page."
    ),
    tools=[fetch_webpage_content],
    policies=[policy.allow_all()], # Auto-approves the python tool execution
    vertex=True,                        # <--- Enable Vertex AI
    project="quiz-generator-499007",      # <--- Your GCP Project ID
    location="us",             # <--- Supported Vertex region
    model="gemini-3.5-flash"
)

# Configure the Reviewer Agent (Quality Check)
reviewer_config = LocalAgentConfig(
    system_instructions=(
        "You are a Quality Check (QC) Agent for customer support.\n"
        "Your task is to review the customer support response drafted by the Writer Agent.\n"
        "Evaluate the draft based on the following criteria:\n"
        "1. Is it accurate and directly answers the customer's query?\n"
        "2. Is the tone polite, professional, and supportive?\n"
        "3. Are there any spelling, grammar, or formatting issues?\n\n"
        "If the response is good, output the approved version. If there are issues, revise and output "
        "the polished, correct response. Output ONLY the final message to be sent to the customer."
    ),
    vertex=True,
    project="quiz-generator-499007",
    location="us",                      # <--- Change from "us-central1" to "us" or "global"
    model="gemini-3.5-flash"
)

## Step 5: Define the Orchestration Pipeline
We create the main asynchronous execution pipeline that manages agent lifecycles, runs chats, and displays results.

In [ ]:
import asyncio

async def run_customer_support_pipeline(url: str, query: str):
    """
    Orchestrates the multi-agent customer support flow.
    """
    print("=" * 60)
    print(f"Customer Query: {query}")
    print(f"Webpage URL:    {url}")
    print("=" * 60 + "\n")

    # 1. Spawn the Support Agent to fetch content and draft response
    print("[1/2] Launching Support Agent...")
    async with Agent(writer_config) as writer:
        prompt = f"Please read the webpage at {url} and answer this customer query: '{query}'"
        writer_response = await writer.chat(prompt)
        draft_text = await writer_response.text()

    print("\n--- Writer Agent Draft ---")
    print(draft_text)
    print("-" * 26 + "\n")

    # 2. Spawn the Reviewer Agent to verify and refine
    print("[2/2] Launching Reviewer Agent (QC)...")
    async with Agent(reviewer_config) as reviewer:
        qc_prompt = (
            f"Please review this draft response to the customer query: '{query}'\n\n"
            f"Draft Response:\n{draft_text}"
        )
        reviewer_response = await reviewer.chat(qc_prompt)
        final_text = await reviewer_response.text()

    print("\n=== FINAL APPROVED CUSTOMER RESPONSE ===")
    print(final_text)
    print("=" * 40 + "\n")

## Step 6: Test the Multi-Agent System
We run a test query asking about the Zen of Python (PEP 20). Since Colab supports top-level `await`, you can run the pipeline directly!

In [ ]:
# Real test inputs
test_url = "https://www.aimletc.com/online-instructor-led-ai-llm-coaching-for-it-technical-professionals/"
test_query = "Who will benefit most from this course?"

# Execute the pipeline
await run_customer_support_pipeline(test_url, test_query)

Customer Query: Who will benefit most from this course?
Webpage URL:    https://www.aimletc.com/online-instructor-led-ai-llm-coaching-for-it-technical-professionals/

[1/2] Launching Support Agent...

--- Writer Agent Draft ---
Hello! Thank you for reaching out with your question about our coaching program. 

According to our course page, the program is designed for **professionals who need to understand, evaluate, and lead AI initiatives**.

The individuals who will benefit most from this course include:
* **Senior IT Architects**
* **Tech Product Managers**
* **Senior Engineers**
* **Tech Entrepreneurs**
* **CTOs**
* **DevOps professionals**

**Prerequisite Note:** 
To get the most out of the course, you should be comfortable reading and understanding Python code (though most of the code for the labs and applications is already written for you).

Please let us know if you have any other questions or if you'd like to request an intro session!

***

### Summary of Work
1. **Fetched Web